In [25]:
import json, numpy as np
from pathlib import Path
from collections import defaultdict, Counter
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle, Arc
from sklearn.cluster import KMeans
from scipy.ndimage import gaussian_filter

REPO_ROOT = Path("../..").resolve()
NOMBRE = "test_27"          # <--- cambia según vídeo
JSON_IN = REPO_ROOT / "data" / "tracking" / f"integrado_{NOMBRE}.json"
OUT_DIR = REPO_ROOT / "outputs" / "tactico"
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(JSON_IN, encoding="utf-8") as fh:
    data = json.load(fh)
frames = data["frames"]
N_FRAMES = data["n_frames"]
color_equipo_bgr = {int(k): tuple(v) for k, v in data["color_equipo_bgr"].items()}
col_eq = {eq: tuple(np.array(bgr[::-1]) / 255.0) for eq, bgr in color_equipo_bgr.items()}

MARGEN = 5.0
def en_campo(c):
    return c is not None and -MARGEN <= c[0] <= 105 + MARGEN and -MARGEN <= c[1] <= 68 + MARGEN

def jugadores_frame(f_idx, equipo, incluir_portero=False):
    roles = ("jugador", "portero") if incluir_portero else ("jugador",)
    return [d["campo"] for d in frames[f_idx]
            if d.get("rol") in roles and d.get("equipo") == equipo
            and en_campo(d.get("campo")) and not d.get("es_fragmento")]

print(f"JSON: {JSON_IN.name} | Frames: {N_FRAMES}")
print(f"Equipos: A={color_equipo_bgr[0]}, B={color_equipo_bgr[1]}")

JSON: integrado_test_27.json | Frames: 750
Equipos: A=(253, 254, 247), B=(121, 241, 254)


In [26]:
# ---- Detectar dirección de ataque automáticamente desde el portero ----
gk = {}
for fr in frames:
    for d in fr:
        if d.get("rol") == "portero" and en_campo(d.get("campo")):
            gk.setdefault(d["equipo"], []).append(d["campo"])

ATAQUE = {}
if gk:
    eq_gk = list(gk.keys())[0]
    x_gk = np.median([p[0] for p in gk[eq_gk]], axis=0)
    lado_gk = "izquierda" if x_gk < 52.5 else "derecha"
    ATAQUE[eq_gk] = "derecha" if lado_gk == "izquierda" else "izquierda"
    ATAQUE[1 - eq_gk] = lado_gk
    print(f"Portero del equipo {eq_gk} en portería {lado_gk} (X={x_gk:.0f})")
else:
    cx = {}
    for eq in (0, 1):
        xs = [d["campo"][0] for fr in frames for d in fr
              if d.get("rol")=="jugador" and d.get("equipo")==eq and en_campo(d.get("campo"))]
        cx[eq] = np.mean(xs) if xs else 52.5
    eq_izq = 0 if cx[0] < cx[1] else 1
    ATAQUE[eq_izq] = "derecha"; ATAQUE[1-eq_izq] = "izquierda"
    print("Sin portero; dirección por centro de masa")

PORTERIA_PROPIA = {eq: (0.0 if ATAQUE[eq]=="derecha" else 105.0) for eq in (0,1)}
print(f"ATAQUE: equipo A={ATAQUE[0]}, equipo B={ATAQUE[1]}")

Portero del equipo 0 en portería izquierda (X=5)
ATAQUE: equipo A=derecha, equipo B=izquierda


In [27]:
# ---- Filtrar fragmentos de tracking: quedarse con jugadores de presencia sólida ----
# El tracking fragmenta algún jugador (un ID se pierde y se crea otro corto). Esos fragmentos
# (pocas detecciones) ensucian las métricas. Se identifican los IDs "titulares" por equipo:
# los que tienen presencia sostenida. Los fragmentos cortos se excluyen del análisis.
from collections import defaultdict

MIN_DET_TITULAR = 200   # un titular aparece en cientos de frames; un fragmento, en pocos

det_por_id_eq = defaultdict(int)
for fr in frames:
    for d in fr:
        if d.get("rol") == "jugador" and d.get("equipo") in (0, 1) and en_campo(d.get("campo")):
            det_por_id_eq[(d["equipo"], d["id"])] += 1

# IDs válidos (titulares) por equipo
ids_validos = {(eq, i) for (eq, i), n in det_por_id_eq.items() if n >= MIN_DET_TITULAR}

print("Jugadores titulares (presencia sólida) por equipo:")
for eq in (0, 1):
    tits = [i for (e, i) in ids_validos if e == eq]
    frags = [i for (e, i), n in det_por_id_eq.items() if e == eq and n < MIN_DET_TITULAR]
    print(f"  Equipo {'AB'[eq]}: {len(tits)} titulares | {len(frags)} fragmentos excluidos {frags if frags else ''}")

# Redefinir jugadores_frame para excluir fragmentos
def jugadores_frame(f_idx, equipo, incluir_portero=False):
    roles = ("jugador", "portero") if incluir_portero else ("jugador",)
    return [d["campo"] for d in frames[f_idx]
            if d.get("rol") in roles and d.get("equipo") == equipo
            and en_campo(d.get("campo")) and (equipo, d["id"]) in ids_validos]

Jugadores titulares (presencia sólida) por equipo:
  Equipo A: 10 titulares | 0 fragmentos excluidos 
  Equipo B: 11 titulares | 0 fragmentos excluidos 


In [28]:
def dibujar_campo(ax, fondo="#2d5a27"):
    ax.add_patch(Rectangle((0,0),105,68, color=fondo, zorder=0))
    bl, lw = "white", 1.5
    ax.add_patch(Rectangle((0,0),105,68, fill=False, edgecolor=bl, linewidth=lw))
    ax.plot([52.5,52.5],[0,68], color=bl, linewidth=lw)
    ax.add_patch(Circle((52.5,34),9.15, fill=False, edgecolor=bl, linewidth=lw))
    for spec in [((0,13.84),16.5,40.32),((88.5,13.84),16.5,40.32),((0,24.84),5.5,18.32),((99.5,24.84),5.5,18.32)]:
        ax.add_patch(Rectangle(spec[0],spec[1],spec[2], fill=False, edgecolor=bl, linewidth=lw))
    ax.set_xlim(-5,110); ax.set_ylim(-5,73); ax.set_aspect("equal"); ax.axis("off")
print("dibujar_campo listo")

dibujar_campo listo


In [29]:
# ---- Centroide, dispersión, amplitud, profundidad por equipo ----
metricas = {0: defaultdict(list), 1: defaultdict(list)}
for f_idx in range(N_FRAMES):
    for eq in (0, 1):
        pts = np.array(jugadores_frame(f_idx, eq))
        if len(pts) < 5: continue
        cen = pts.mean(axis=0)
        metricas[eq]["cx"].append(cen[0]); metricas[eq]["cy"].append(cen[1])
        metricas[eq]["disp"].append(np.linalg.norm(pts-cen, axis=1).std())
        metricas[eq]["amp"].append(pts[:,1].max()-pts[:,1].min())
        metricas[eq]["prof"].append(pts[:,0].max()-pts[:,0].min())

print("=== MÉTRICAS DE BLOQUE ===")
for eq in (0,1):
    print(f"Equipo {'AB'[eq]}: centroide=({np.mean(metricas[eq]['cx']):.0f},{np.mean(metricas[eq]['cy']):.0f}) | "
          f"dispersión={np.mean(metricas[eq]['disp']):.1f}m | amplitud={np.mean(metricas[eq]['amp']):.0f}m | "
          f"profundidad={np.mean(metricas[eq]['prof']):.0f}m")

fig, ax = plt.subplots(figsize=(11,8), dpi=100)
dibujar_campo(ax)
for eq in (0,1):
    cx, cy = np.mean(metricas[eq]["cx"]), np.mean(metricas[eq]["cy"])
    ax.scatter([cx],[cy], s=300, c=[col_eq[eq]], edgecolors="black", linewidth=2, zorder=6)
    ax.add_patch(Circle((cx,cy), np.mean(metricas[eq]["disp"]), fill=False, edgecolor=col_eq[eq], linewidth=2, linestyle="--"))
    ax.annotate("AB"[eq], (cx,cy), fontsize=14, fontweight="bold", ha="center", va="center")
ax.set_title("Centroide y dispersión media", color="white", fontsize=13)
fig.patch.set_facecolor("#1a1a1a")
plt.savefig(OUT_DIR/f"centroide_{NOMBRE}.png", dpi=110, bbox_inches="tight", facecolor="#1a1a1a"); plt.show()
print(f"Guardado: centroide_{NOMBRE}.png")

=== MÉTRICAS DE BLOQUE ===
Equipo A: centroide=(47,43) | dispersión=6.0m | amplitud=38m | profundidad=27m
Equipo B: centroide=(54,41) | dispersión=6.6m | amplitud=41m | profundidad=34m
Guardado: centroide_test_27.png


C:\Users\sgghu\AppData\Local\Temp\ipykernel_32744\780952792.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(OUT_DIR/f"centroide_{NOMBRE}.png", dpi=110, bbox_inches="tight", facecolor="#1a1a1a"); plt.show()


In [30]:
# ---- Mapas de calor por equipo ----
def mapa_calor(equipo, bins=(52,34), sigma=1.5):
    pts = np.array([d["campo"] for fr in frames for d in fr
                    if d.get("rol")=="jugador" and d.get("equipo")==equipo
                    and en_campo(d.get("campo")) and not d.get("es_fragmento")])
    H,_,_ = np.histogram2d(pts[:,0], pts[:,1], bins=bins, range=[[0,105],[0,68]])
    return gaussian_filter(H.T, sigma=sigma)

fig, axes = plt.subplots(1,2, figsize=(20,7), dpi=100)
for eq, ax in zip((0,1), axes):
    dibujar_campo(ax, fondo="#1a3a17")
    ax.imshow(mapa_calor(eq), extent=[0,105,0,68], origin="lower", cmap="hot", alpha=0.6, aspect="equal", zorder=1)
    ax.set_title(f"Mapa de calor - Equipo {'AB'[eq]}", color="white", fontsize=13)
fig.patch.set_facecolor("#1a1a1a")
plt.savefig(OUT_DIR/f"calor_{NOMBRE}.png", dpi=110, bbox_inches="tight", facecolor="#1a1a1a"); plt.show()
print(f"Guardado: calor_{NOMBRE}.png")

Guardado: calor_test_27.png


C:\Users\sgghu\AppData\Local\Temp\ipykernel_32744\1450040728.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(OUT_DIR/f"calor_{NOMBRE}.png", dpi=110, bbox_inches="tight", facecolor="#1a1a1a"); plt.show()


In [31]:
# ---- Posesión con NEUTRO (dist_control=10, franja=1.0) ----
posesion = {0:0, 1:0, "neutro":0}
zona_posesion = {0:[0,0,0], 1:[0,0,0]}
for f_idx in range(N_FRAMES):
    bpos = next((d["campo"] for d in frames[f_idx] if d.get("rol")=="balon" and en_campo(d.get("campo"))), None)
    if bpos is None: continue
    dmin = {0:999, 1:999}
    for d in frames[f_idx]:
        if d.get("rol")=="jugador" and d.get("equipo") in (0,1) and en_campo(d.get("campo")) and not d.get("es_fragmento"):
            dd = np.hypot(d["campo"][0]-bpos[0], d["campo"][1]-bpos[1])
            dmin[d["equipo"]] = min(dmin[d["equipo"]], dd)
    d0,d1 = dmin[0], dmin[1]
    if min(d0,d1)>10 or abs(d0-d1)<1.0:
        posesion["neutro"]+=1
    else:
        eq = 0 if d0<d1 else 1; posesion[eq]+=1
        tercio = 0 if bpos[0]<35 else (1 if bpos[0]<70 else 2)
        zona_posesion[eq][tercio]+=1

total = sum(posesion.values())
print("=== POSESIÓN ===")
print(f"A: {100*posesion[0]/total:.0f}% | Neutro: {100*posesion['neutro']/total:.0f}% | B: {100*posesion[1]/total:.0f}%")

col_a = tuple(np.array(color_equipo_bgr[0][::-1])/255.0); col_b = tuple(np.array(color_equipo_bgr[1][::-1])/255.0)
fig,(ax1,ax2) = plt.subplots(1,2, figsize=(16,6), dpi=100)
ax1.pie([posesion[0],posesion["neutro"],posesion[1]], labels=["A","Neutro","B"],
        colors=[col_a,"#a0a0a0",col_b], autopct="%1.0f%%", textprops={"fontsize":12})
ax1.set_title("Posesión", fontsize=13)
x=np.arange(3); w=0.35
ax2.bar(x-w/2, zona_posesion[0], w, label="A", color=col_a, edgecolor="black")
ax2.bar(x+w/2, zona_posesion[1], w, label="B", color=col_b, edgecolor="black")
ax2.set_xticks(x); ax2.set_xticklabels(["Defensa","Medio","Ataque"]); ax2.set_title("Por zona"); ax2.legend()
plt.savefig(OUT_DIR/f"posesion_{NOMBRE}.png", dpi=110, bbox_inches="tight"); plt.show()
print(f"Guardado: posesion_{NOMBRE}.png")

=== POSESIÓN ===
A: 25% | Neutro: 13% | B: 62%
Guardado: posesion_test_27.png


C:\Users\sgghu\AppData\Local\Temp\ipykernel_32744\924492288.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(OUT_DIR/f"posesion_{NOMBRE}.png", dpi=110, bbox_inches="tight"); plt.show()


In [32]:
# ---- Formación: ambos equipos enfrentados, porteros en sus porterías ----
CATALOGO = {"4-4-2":[4,4,2],"4-3-3":[4,3,3],"4-2-3-1":[4,2,3,1],"4-5-1":[4,5,1],
            "4-1-4-1":[4,1,4,1],"4-4-1-1":[4,4,1,1],"3-5-2":[3,5,2],"3-4-3":[3,4,3],
            "5-3-2":[5,3,2],"5-4-1":[5,4,1]}

def formacion_equipo(eq):
    det = defaultdict(list)
    for fr in frames:
        for d in fr:
            if d.get("rol")=="jugador" and d.get("equipo")==eq and en_campo(d.get("campo")) and not d.get("es_fragmento"):
                det[d["id"]].append(d["campo"])
    jug = {i: np.median(ps, axis=0) for i,ps in det.items() if len(ps)>=20}
    if len(jug)<8: return None, None
    ids=list(jug); P=np.array([jug[i] for i in ids])
    avance = P[:,0] if ATAQUE[eq]=="derecha" else (105-P[:,0])
    orden=np.argsort(avance); n=len(jug)
    mejor,mc=None,1e9
    for nom,pat in CATALOGO.items():
        if sum(pat)!=n: continue
        coste,idx,cen=0,0,[]
        for c in pat:
            g=orden[idx:idx+c]; cen.append(avance[g].mean()); coste+=avance[g].std()*c; idx+=c
        if any(cen[i]>=cen[i+1] for i in range(len(cen)-1)): coste+=50
        if coste<mc: mc,mejor=coste,(nom,pat)
    return jug, mejor

def dibujar_campo_c(ax):
    ax.add_patch(Rectangle((0,0),105,68, color="#2d5a27", zorder=0)); bl,lw="white",1.5
    ax.add_patch(Rectangle((0,0),105,68, fill=False, edgecolor=bl, linewidth=lw))
    ax.plot([52.5,52.5],[0,68], color=bl, linewidth=lw); ax.add_patch(Circle((52.5,34),9.15, fill=False, edgecolor=bl, linewidth=lw))
    for s in [((0,13.84),16.5,40.32),((88.5,13.84),16.5,40.32),((0,24.84),5.5,18.32),((99.5,24.84),5.5,18.32)]:
        ax.add_patch(Rectangle(s[0],s[1],s[2], fill=False, edgecolor=bl, linewidth=lw))
    ax.set_xlim(-5,110); ax.set_ylim(-5,73); ax.set_aspect("equal"); ax.axis("off")

def pos_ideales(formacion, ataca):
    patron = CATALOGO[formacion]; n_lin=len(patron)
    if ataca=="derecha":
        gk_x=5; xs=np.linspace(20,50,n_lin)
    else:
        gk_x=100; xs=np.linspace(85,55,n_lin)
    out=[("GK",gk_x,34)]
    for lx,c in zip(xs,patron):
        ys = np.linspace(10,58,c+2)[1:-1] if c>1 else [34]
        for y in ys: out.append(("J",lx,y))
    return out

fig,(ax_real,ax_ideal)=plt.subplots(2,1, figsize=(13,18), dpi=100)
dibujar_campo_c(ax_real)
formaciones={}
for eq in (0,1):
    jug, res = formacion_equipo(eq); formaciones[eq]=res
    col = col_eq[eq]
    if jug:
        for i,pos in jug.items():
            ax_real.scatter(pos[0],pos[1], s=220, c=[col], edgecolors="black", linewidth=1.5, zorder=5)
    gk = next((d["campo"] for fr in frames for d in fr if d.get("rol")=="portero" and d.get("equipo")==eq and en_campo(d.get("campo"))), None)
    if gk:
        ax_real.scatter(gk[0],gk[1], s=240, c=[col], edgecolors="white", linewidth=2.5, zorder=6)
    else:
        gx = 5 if ATAQUE[eq]=="derecha" else 100
        ax_real.scatter(gx,34, s=240, c=[col], edgecolors="yellow", linewidth=2.5, zorder=6, marker="s")
        ax_real.annotate("GK (hip.)", (gx,34), fontsize=9, color="yellow", ha="center", va="bottom")
txt=" vs ".join(f"{'AB'[eq]} 1-{formaciones[eq][0]}" for eq in (0,1) if formaciones[eq])
ax_real.set_title(f"Posiciones reales — {txt}", color="white", fontsize=14)

dibujar_campo_c(ax_ideal)
for eq in (0,1):
    if not formaciones[eq]: continue
    col = col_eq[eq]
    for tipo,x,y in pos_ideales(formaciones[eq][0], ATAQUE[eq]):
        edge = "white" if tipo=="J" else "yellow"
        ax_ideal.scatter(x,y, s=240, c=[col], edgecolors=edge, linewidth=2, zorder=6)
ax_ideal.set_title(f"Formación ideal enfrentada — {txt}", color="white", fontsize=14)
fig.patch.set_facecolor("#1a1a1a")
plt.savefig(OUT_DIR/f"formacion_{NOMBRE}.png", dpi=110, bbox_inches="tight", facecolor="#1a1a1a"); plt.show()
for eq in (0,1):
    if formaciones[eq]: print(f"Equipo {'AB'[eq]}: 1-{formaciones[eq][0]}, ataca {ATAQUE[eq]}")
print(f"Guardado: formacion_{NOMBRE}.png")

Equipo A: 1-3-5-2, ataca derecha
Guardado: formacion_test_27.png


C:\Users\sgghu\AppData\Local\Temp\ipykernel_32744\4279031775.py:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(OUT_DIR/f"formacion_{NOMBRE}.png", dpi=110, bbox_inches="tight", facecolor="#1a1a1a"); plt.show()
